# Pretrained ITSELF + learned reranker on RSTPReid

Attach only the `tbps-benchmark` dataset, enable Internet and GPU, then run cells in order in a fresh Kaggle session. No Kaggle `rstp.pth` model input is needed.

The notebook downloads `GenAI4ELab/itself-rstp/best.pth` (about 1.91 GB) at a fixed revision. ITSELF is already trained; both the image/text encoder and reranker are fine-tuned here. The Hugging Face repository contains no config or sampler information, so this notebook does not claim that its checkpoint used the Identity sampler.

Candidate retrieval uses the checkpoint's global CLIP encoder, with direct resize to 384x128. Full ITSELF global+GRAB scoring is not included. Train/val are used for joint training/checkpoint selection; final evaluation uses test.


## 1. Install the project

In [ ]:
import subprocess, sys
from pathlib import Path
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '-q',
    'git+https://github.com/lqb464/clip-person-search.git', 'huggingface_hub'
], check=True)


## 2. Locate RSTPReid and check GPU

In [ ]:
from collections import Counter
import json, torch
INPUT_DIR = Path('/kaggle/input')
annotations = [p for p in INPUT_DIR.rglob('data_captions.json') if p.parent.name.lower() == 'rstpreid']
if len(annotations) != 1:
    raise RuntimeError(f'Expected one RSTPReid annotation file; found {annotations}')
RSTP_ROOT = annotations[0].parent
if not (RSTP_ROOT / 'imgs').is_dir():
    raise FileNotFoundError(f'No imgs/ under {RSTP_ROOT}')
records = json.loads(annotations[0].read_text(encoding='utf-8'))
print('RSTP root:', RSTP_ROOT)
print('Splits:', Counter(r.get('split') for r in records))
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before training.')
print('GPU:', torch.cuda.get_device_name(0))


## 3. Download the pretrained ITSELF checkpoint

In [ ]:
from huggingface_hub import hf_hub_download
HF_MODEL_ID = 'GenAI4ELab/itself-rstp'
HF_REVISION = 'd575c9691832b0a3e88d076faeba9973260956ee'
ITSELF_CHECKPOINT = hf_hub_download(
    repo_id=HF_MODEL_ID,
    filename='best.pth',
    revision=HF_REVISION,
    cache_dir='/kaggle/working/huggingface',
)
print('Downloaded ITSELF checkpoint:', ITSELF_CHECKPOINT)


## 4. Check checkpoint loading and global/token feature extraction

Stop here if this cell reports a checkpoint incompatibility; do not start training until it passes.

In [ ]:
# Check all required encoder keys/shapes before starting the long training stage.
import gc
from person_search.model import load_model, encode_images, encode_text, encode_image_tokens, encode_text_tokens
from PIL import Image
model, preprocess, tokenizer, device = load_model(
    checkpoint=ITSELF_CHECKPOINT, model_name='ViT-B-16',
    image_size=(384, 128), image_preprocess='itself',
)
with torch.no_grad():
    record = next(r for r in records if r['split'] == 'train')
    with Image.open(RSTP_ROOT / 'imgs' / record['img_path']) as image:
        image = image.convert('RGB')
        image_features = encode_images(model, preprocess, [image], device)
        image_tokens, image_mask = encode_image_tokens(model, preprocess, [image], device)
    text_features = encode_text(model, tokenizer, [str(record['captions'][0])], device)
    text_tokens, text_mask = encode_text_tokens(model, tokenizer, [str(record['captions'][0])], device)
print('Global features:', image_features.shape, text_features.shape)
print('Token features:', image_tokens.shape, text_tokens.shape)
del model, image_features, text_features, image_tokens, text_tokens, image_mask, text_mask
gc.collect()
torch.cuda.empty_cache()


## 5. Jointly fine-tune the encoder and reranker

Both image and text encoders receive gradients. Encoder LR is 1e-6 and matcher LR is 2e-4. Batch size is reduced to 4 for the T4; lower to 2 if needed. The selected checkpoint saves both models together. Validation loss selects the checkpoint.

In [ ]:
RERANKER_CHECKPOINT = '/kaggle/working/rstpreid-itself-joint-reranker.pth'
subprocess.run([
    sys.executable, '-m', 'person_search.cli', 'train-reranker-rstpreid',
    '--root', str(RSTP_ROOT),
    '--clip-checkpoint', str(ITSELF_CHECKPOINT),
    '--clip-model', 'ViT-B-16',
    '--image-size', '384', '128',
    '--image-preprocess', 'itself',
    '--output', RERANKER_CHECKPOINT,
    '--batch-size', '4',
    '--epochs', '5',
    '--encoder-learning-rate', '1e-6',
    '--learning-rate', '2e-4',
    '--num-workers', '2',
], check=True)

## 6. Evaluate baseline and top-100 reranking on test

Outputs include R@1/5/10, mAP, metric deltas, harmed-query counts, and a per-query CSV.

The baseline and reranked metrics in this report both use the fine-tuned encoder restored from the joint checkpoint. Compare the separate original baseline report as well, since encoder fine-tuning can change global retrieval quality.

In [ ]:
ORIGINAL_METRICS_PATH = '/kaggle/working/rstpreid-original-itself-metrics.json'
subprocess.run([
    sys.executable, '-m', 'person_search.cli', 'evaluate-rstpreid',
    '--root', str(RSTP_ROOT),
    '--clip-checkpoint', str(ITSELF_CHECKPOINT),
    '--clip-model', 'ViT-B-16',
    '--image-size', '384', '128',
    '--image-preprocess', 'itself',
    '--split', 'test',
    '--batch-size', '16',
    '--output', ORIGINAL_METRICS_PATH,
], check=True)

METRICS_PATH = '/kaggle/working/rstpreid-itself-rerank-metrics.json'
PER_QUERY_PATH = '/kaggle/working/rstpreid-itself-rerank-per-query.csv'
subprocess.run([
    sys.executable, '-m', 'person_search.cli', 'evaluate-rstpreid',
    '--root', str(RSTP_ROOT),
    '--clip-checkpoint', str(ITSELF_CHECKPOINT),
    '--clip-model', 'ViT-B-16',
    '--image-size', '384', '128',
    '--image-preprocess', 'itself',
    '--reranker-checkpoint', RERANKER_CHECKPOINT,
    '--rerank-top-m', '100',
    '--split', 'test',
    '--batch-size', '16',
    '--output', METRICS_PATH,
    '--per-query-output', PER_QUERY_PATH,
], check=True)
print(Path(METRICS_PATH).read_text())
print('Per-query details:', PER_QUERY_PATH)